In [ ]:
import pandas as pd

# Define the file path
file_path = '/home/cristic/data/Bgeorge/mcd_rppg/snapshots/929fb19c5ff2b5c8ed64a7c3a123744346674e88/db.csv'

# Load the CSV file into a DataFrame
df = pd.read_csv(file_path)

# Filter rows where patient_id contains the text "6082"
# .astype(str) ensures it handles numbers and text properly
# na=False ignores any blank/empty cells in that column
filtered_df = df[df['patient_id'].astype(str).str.contains('6082', na=False)]

# Display all matching values/rows in the notebook
filtered_df

In [ ]:
print(vitals_dict.get("6082_IriunWebcam_after", {}))

In [ ]:
video_name = "6082_IriunWebcam_after.avi"
db_key = "6082_IriunWebcam_after"
subject_vitals = vitals_dict.get(db_key, {})
print(f"Vitals for {db_key}: {subject_vitals}")

In [ ]:
# %% [markdown]
"""
## Debug Single File Cell
Comprehensive debugging for a single video file to verify:
1. File paths from db.csv
2. Meta/PPG sync alignment
3. PPG signal quality
4. HR calculation
"""

# %%
def debug_single_file(video_name, db_key=None):
    """
    Debug a single video file with all available information.
    If db_key is None, it will be inferred from video_name.
    """
    print(f"\n{'='*70}")
    print(f"DEBUGGING SINGLE FILE: {video_name}")
    print('='*70)

    # --- 1. Get vitals and paths from db.csv ---
    if db_key is None:
        parts = os.path.splitext(video_name)[0].split('_')
        patient_id = parts[0]
        camera = '_'.join(parts[1:-1])
        effort = parts[-1] if parts[-1] in ['before', 'after'] else 'after'
        db_key = f"{patient_id}_{camera}_{effort}"

    subject_vitals = vitals_dict.get(db_key, {})
    print(f"\nDB Key: {db_key}")
    print(f"Subject Vitals: {subject_vitals}")

    if not subject_vitals:
        print("❌ No matching entry in db.csv for this video!")
        return

    # --- 2. Verify file paths ---
    print("\nFile Paths from db.csv:")
    for file_type in ['video_path', 'meta_path', 'ppg_sync_path', 'ppg_path']:
        path = subject_vitals.get(file_type, 'NOT FOUND')
        full_path = os.path.join(SNAPSHOT_DIR, path) if path != 'NOT FOUND' else path
        exists = os.path.exists(full_path) if path != 'NOT FOUND' else False
        print(f"  {file_type}: {path} (Exists: {exists})")

    # --- 3. Load and verify meta file ---
    print("\nMeta File Analysis:")
    meta_map, meta_path = load_meta_file(video_name)
    if meta_map:
        print(f"  Meta path: {meta_path}")
        print(f"  Total frames: {len(meta_map)}")
        print(f"  First 5 frames:")
        for frame, ts in list(meta_map.items())[:5]:
            print(f"    Frame {frame}: {ts}")
    else:
        print("  ❌ No meta file found!")

    # --- 4. Load and verify PPG sync file ---
    print("\nPPG Sync File Analysis:")
    sync_map, sync_path = load_ppg_sync(video_name)
    if sync_map:
        print(f"  Sync path: {sync_path}")
        print(f"  Total sync entries: {len(sync_map)}")
        print(f"  First 5 entries:")
        for frame, amp in list(sync_map.items())[:5]:
            print(f"    Frame {frame}: {amp}")
    else:
        print("  ❌ No PPG sync file found!")

    # --- 5. Load and verify PPG raw file ---
    print("\nPPG Raw File Analysis:")
    ppg_raw = load_ppg_raw(video_name)
    if ppg_raw:
        print(f"  Total PPG samples: {len(ppg_raw)}")
        print(f"  First 5 samples:")
        for i, (ts, amp) in enumerate(ppg_raw[:5]):
            print(f"    Sample {i}: {ts} -> {amp}")
    else:
        print("  ❌ No PPG raw file found!")

    # --- 6. Test PPG alignment for first chunk ---
    print("\nTesting PPG Alignment for Chunk 0-450:")
    ppg_signal, fs, sync_info = get_ppg_for_video_chunk(video_name, 0, 450, subject_vitals)
    if ppg_signal is not None:
        print(f"  PPG signal length: {len(ppg_signal)}")
        print(f"  Sampling rate: {fs} Hz")
        print(f"  Non-zero samples: {np.sum(ppg_signal != 0)}")
        print(f"  PPG mean: {np.mean(ppg_signal):.4f}, std: {np.std(ppg_signal):.4f}")
        print(f"  Pulse from db.csv: {subject_vitals.get('pulse', 'NOT FOUND')}")

        # Plot PPG signal
        plt.figure(figsize=(12, 4))
        plt.plot(ppg_signal, color='orange', linewidth=1)
        plt.title(f"PPG Signal for {video_name} (Chunk 0-450) | fs={fs} Hz")
        plt.xlabel("Sample Index")
        plt.ylabel("Normalized Amplitude")
        plt.grid(True, linestyle=':', alpha=0.5)
        plt.show()

        # Calculate HR
        hr = calculate_hr(ppg_signal, fs=fs, subject_pulse=subject_vitals.get('pulse'))
        print(f"\n  Calculated HR: {hr:.1f} BPM")

        # Plot FFT if HR seems off
        if hr < 40 or hr > 120:
            print(f"  ⚠️ HR seems unusual ({hr:.1f} BPM), plotting FFT...")
            n = len(ppg_signal)
            yf = fft(ppg_signal)
            xf = fftfreq(n, 1/fs)[:n//2]
            magnitude = 2.0/n * np.abs(yf[0:n//2])

            plt.figure(figsize=(12, 4))
            plt.plot(xf, magnitude)
            plt.axvline(x=hr/60.0, color='red', linestyle='--', label=f'{hr:.1f} BPM')
            plt.xlabel("Frequency (Hz)")
            plt.ylabel("Magnitude")
            plt.title(f"FFT for {video_name} | True HR should be ~{subject_vitals.get('pulse', '?')} BPM")
            plt.legend()
            plt.grid(True)
            plt.show()
    else:
        print("  ❌ Failed to get PPG signal!")

    # --- 7. Test ROI extraction for first frame ---
    print("\nTesting ROI Extraction for First Frame:")
    landmarker_instance = init_landmarker()
    if landmarker_instance:
        roi_patches, frame_count, original_frames, first_frame = process_video_chunk(
            os.path.join(SNAPSHOT_DIR, "video", video_name), 0, 1, landmarker_instance)

        if first_frame is not None:
            print(f"  Frame extracted: {first_frame.shape}")
            print(f"  ROI patches shape: {roi_patches.shape if len(roi_patches) > 0 else 'None'}")

            # Show first frame with ROIs
            if len(roi_patches) > 0:
                frame_copy = first_frame.copy()
                h, w = frame_copy.shape[:2]
                rgb_frame = cv2.cvtColor(frame_copy, cv2.COLOR_BGR2RGB)
                mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
                detection_result = landmarker_instance.detect(mp_image)

                if detection_result.face_landmarks:
                    landmarks = detection_result.face_landmarks[0]
                    for idx, group in ROI_LANDMARK_GROUPS.items():
                        try:
                            pts = [[int(landmarks[lm_idx].x * w), int(landmarks[lm_idx].y * h)] for lm_idx in group]
                            pts = np.array(pts)
                            xmin, ymin = np.min(pts, axis=0)
                            xmax, ymax = np.max(pts, axis=0)
                            cv2.rectangle(frame_copy, (xmin, ymin), (xmax, ymax), (0, 255, 0), 2)
                            cv2.putText(frame_copy, PATCH_LABELS[idx], (xmin, ymin-10),
                                      cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
                        except:
                            continue

                plt.figure(figsize=(10, 8))
                plt.imshow(cv2.cvtColor(frame_copy, cv2.COLOR_BGR2RGB))
                plt.title(f"First Frame with ROIs | {video_name}")
                plt.axis('off')
                plt.show()
        else:
            print("  ❌ Failed to extract first frame!")
    else:
        print("  ❌ Failed to initialize landmarker!")

# Example usage:
# debug_single_file("6082_IriunWebcam_after.avi")
# debug_single_file("1020_FullHDwebcam_after.avi")

# MCG\_RPPG Preprocessing - Complete Jupyter Notebook

**Timestamp-Based Alignment for Multi-Camera PPG Data**

---

## 📌 Overview

This notebook preprocesses **video and PPG data** from the [MCG\_RPPG dataset](https://github.com/ksyegorov/mcd_rppg) to:

- Extract **8 facial ROI patches** (forehead, cheeks, jaw, nose, chin).
- Align **PPG signals** with video frames using **timestamps** (from `meta/` and `ppg_sync/` files).
- Calculate **heart rate (HR)** using **peak detection** (20–220 BPM).
- Save processed chunks as `.npz` files for model training.

**Key Features:**

- ✅ Uses **real PPG data** (no synthetic signals).
- ✅ **Timestamp-based alignment** (handles sparse `ppg_sync/` and `.PW` files).
- ✅ **CPU-only processing** (avoids CUDA initialization errors).
- ✅ **20 workers** for parallel processing.
- ✅ **Valid HR range: 20–220 BPM**.
- ✅ **Full vitals** from `db.csv` (saturation, temperature, hemoglobin, etc.).
- ✅ **Multi-camera support** (FullHDwebcam, IriunWebcam, USBVideo).

---

## 📂 Directory Structure

```
SNAPSHOT_DIR/
├── video/            # Input videos (e.g., 1020_FullHDwebcam_after.avi)
├── meta/             # Frame timestamps (e.g., meta/1020_FullHDwebcam_before.txt)
├── ppg_sync/         # Sparse PPG amplitudes (e.g., ppg_sync/1020_IriunWebcam_before.txt)
├── ppg/              # Raw PPG samples (e.g., ppg/1020_before.PW)
└── db.csv            # Subject vitals (patient_id, age, sex, etc.)

OUTPUT_DIR/
├── processed_data/   # Output: .npz files (ROI patches, PPG, HR, vitals)
└── roi_extraction.py # Standalone ROI extraction script
```

In [ ]:
###New one

In [ ]:
# %% [markdown]
"""
## 1. Setup and Imports
"""

# %%
import os
import cv2
import numpy as np
import pandas as pd
import glob
import subprocess
import mediapipe as mp
from scipy.signal import butter, filtfilt, find_peaks
from scipy.fft import fft, fftfreq
import matplotlib.pyplot as plt
from multiprocessing import Pool
import warnings
import json
from tqdm.notebook import tqdm
from datetime import datetime
import traceback

warnings.filterwarnings("ignore")
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 4)
os.environ["CUDA_VISIBLE_DEVICES"] = ""

# %% [markdown]
"""
### Configuration
"""

# %%
SNAPSHOT_DIR = "/home/cristic/data/Bgeorge/mcd_rppg/snapshots/929fb19c5ff2b5c8ed64a7c3a123744346674e88/"
OUTPUT_DIR = "/home/cristic/data/processed/mistral2/"
MEDIAPIPE_MODEL_PATH = "/home/cristic/face_landmarker.task"
DB_PATH = os.path.join(SNAPSHOT_DIR, "db.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "processed_data"), exist_ok=True)

ROI_LANDMARK_GROUPS = {
    0: [70, 71, 139, 156], 1: [300, 301, 368, 383],
    2: [117, 118, 101, 50], 3: [346, 347, 330, 280],
    4: [205, 206, 207, 187], 5: [425, 426, 427, 411],
    6: [6, 197, 195, 5], 7: [199, 200, 18, 42]
}

PATCH_LABELS = {
    0: "Left_Forehead", 1: "Right_Forehead", 2: "Left_Cheek",
    3: "Right_Cheek", 4: "Left_Jaw", 5: "Right_Jaw", 6: "Nose", 7: "Chin"
}

VIDEO_FPS = 30.0
CHUNK_SIZE = 450
TARGET_PPG_LEN = CHUNK_SIZE  # FIXED: 450 samples
PATCH_SIZE = (32, 32)
MARGIN_RATIO = 0.1
LOWCUT = 0.5
HIGHCUT = 3.5
ORDER = 4
MIN_HR = 20.0
MAX_HR = 220.0
CHUNKS_PER_VIDEO = 4
TEST_MODE = False
NUM_WORKERS = 20
MAX_TEST_SAMPLES=5
print("Configuration loaded. PPG samples per chunk:", TARGET_PPG_LEN)

In [ ]:
# %% [markdown]
"""
## 2. Initialization
"""

# %%
def init_landmarker():
    try:
        return mp.tasks.vision.FaceLandmarker.create_from_options(
            mp.tasks.vision.FaceLandmarkerOptions(
                base_options=mp.tasks.BaseOptions(model_asset_path=MEDIAPIPE_MODEL_PATH),
                running_mode=mp.tasks.vision.RunningMode.IMAGE,
                num_faces=1
            )
        )
    except Exception as e:
        print(f"Error initializing MediaPipe: {str(e)}")
        return None

def load_vitals(db_path):
    try:
        df = pd.read_csv(db_path)
        vitals_dict = {}
        for _, row in df.iterrows():
            patient_id = str(int(row['patient_id']))
            video_path = row.get('video', '')
            if video_path:
                parts = os.path.splitext(os.path.basename(video_path))[0].split('_')
                camera = '_'.join(parts[1:-1]) if len(parts) > 2 else ''
                effort = parts[-1] if parts[-1] in ['before', 'after'] else 'after'
                key = f"{patient_id}_{camera}_{effort}"
            else:
                key = patient_id
            vitals_dict[key] = {
                'age': row.get('age'), 'sex': row.get('sex'), 'bmi': row.get('bmi'),
                'saturation': row.get('saturation'), 'temperature': row.get('temperature'),
                'hemoglobin': row.get('hemoglobin'), 'pulse': row.get('pulse'),
                'video_path': row.get('video'), 'meta_path': row.get('meta'),
                'ppg_sync_path': row.get('ppg_sync'), 'ppg_path': row.get('ppg')
            }
        return vitals_dict
    except Exception as e:
        print(f"Error loading vitals: {str(e)}")
        return {}

def parse_timestamp(ts_str):
    return datetime.strptime(ts_str, "%Y-%m-%d %H:%M:%S.%f")

vitals_dict = load_vitals(DB_PATH)
landmarker = init_landmarker()
print(f"Loaded vitals for {len(vitals_dict)} subjects")
print(f"MediaPipe initialized: {landmarker is not None}")

In [ ]:
# %% [markdown]
"""
## 3. Data Loading Functions
"""

# %%
def load_meta_file(video_name):
    base_name = os.path.splitext(video_name)[0]
    parts = base_name.split('_')
    patient_id = parts[0]
    camera = '_'.join(parts[1:-1]) if len(parts) > 2 else ''
    effort = parts[-1] if parts[-1] in ['before', 'after'] else 'after'
    db_key = f"{patient_id}_{camera}_{effort}"
    if db_key in vitals_dict and vitals_dict[db_key].get('meta_path'):
        meta_path = os.path.join(SNAPSHOT_DIR, vitals_dict[db_key]['meta_path'])
        if os.path.exists(meta_path):
            meta_map = {}
            with open(meta_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 3:
                        try:
                            meta_map[int(parts[0])] = parse_timestamp(" ".join(parts[1:]))
                        except ValueError:
                            continue
            return meta_map, meta_path
    return {}, None

def load_ppg_sync(video_name):
    base_name = os.path.splitext(video_name)[0]
    parts = base_name.split('_')
    patient_id = parts[0]
    camera = '_'.join(parts[1:-1]) if len(parts) > 2 else ''
    effort = parts[-1] if parts[-1] in ['before', 'after'] else 'after'
    db_key = f"{patient_id}_{camera}_{effort}"
    if db_key in vitals_dict and vitals_dict[db_key].get('ppg_sync_path'):
        sync_path = os.path.join(SNAPSHOT_DIR, vitals_dict[db_key]['ppg_sync_path'])
        if os.path.exists(sync_path):
            sync_map = {}
            with open(sync_path, 'r') as f:
                for line_num, line in enumerate(f):
                    line = line.strip()
                    if line:
                        try:
                            sync_map[line_num] = float(line.split()[0])
                        except (ValueError, IndexError):
                            continue
            return sync_map, sync_path
    return {}, None

def load_ppg_raw(video_name):
    base_name = os.path.splitext(video_name)[0]
    parts = base_name.split('_')
    patient_id = parts[0]
    camera = '_'.join(parts[1:-1]) if len(parts) > 2 else ''
    effort = parts[-1] if parts[-1] in ['before', 'after'] else 'after'
    db_key = f"{patient_id}_{camera}_{effort}"
    if db_key in vitals_dict and vitals_dict[db_key].get('ppg_path'):
        ppg_path = os.path.join(SNAPSHOT_DIR, vitals_dict[db_key]['ppg_path'])
        if os.path.exists(ppg_path):
            ppg_samples = []
            with open(ppg_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 3:
                        try:
                            ppg_samples.append((parse_timestamp(" ".join(parts[1:])), float(parts[0])))
                        except ValueError:
                            continue
            return ppg_samples
    return None

In [ ]:
# %% [markdown]
"""
## 4. Video Processing and ROI Extraction
"""

# %%
def get_video_info(video_path):
    try:
        cmd = ['ffprobe', '-v', 'error', '-select_streams', 'v:0',
               '-show_entries', 'stream=nb_frames,r_frame_rate', '-of', 'json', video_path]
        result = subprocess.run(cmd, capture_output=True, text=True)
        info = json.loads(result.stdout)
        return int(info['streams'][0]['nb_frames']), float(eval(info['streams'][0]['r_frame_rate']))
    except:
        return 10000, 30.0

def extract_rois(frame, landmarks, h, w):
    roi_patches = np.zeros((8, PATCH_SIZE[1], PATCH_SIZE[0], 3), dtype=np.uint8)
    for idx, group in ROI_LANDMARK_GROUPS.items():
        try:
            pts = [[int(landmarks[lm_idx].x * w), int(landmarks[lm_idx].y * h)] for lm_idx in group]
            pts = np.array(pts)
            xmin, ymin = np.min(pts, axis=0)
            xmax, ymax = np.max(pts, axis=0)
            box_w, box_h = xmax - xmin, ymax - ymin
            margin_x, margin_y = int(box_w * MARGIN_RATIO), int(box_h * MARGIN_RATIO)
            xmin, xmax = max(0, xmin - margin_x), min(w-1, xmax + margin_x)
            ymin, ymax = max(0, ymin - margin_y), min(h-1, ymax + margin_y)
            if (xmax > xmin) and (ymax > ymin):
                crop = frame[ymin:ymax, xmin:xmax]
                roi_patches[idx] = cv2.resize(crop, PATCH_SIZE, interpolation=cv2.INTER_LINEAR)
            else:
                center_x, center_y = w // 2, h // 2
                crop = frame[center_y-16:center_y+16, center_x-16:center_x+16]
                roi_patches[idx] = cv2.resize(crop, PATCH_SIZE, interpolation=cv2.INTER_LINEAR)
        except:
            center_x, center_y = w // 2, h // 2
            crop = frame[center_y-16:center_y+16, center_x-16:center_x+16]
            roi_patches[idx] = cv2.resize(crop, PATCH_SIZE, interpolation=cv2.INTER_LINEAR)
    return roi_patches

def process_video_chunk(video_path, start_frame, end_frame, landmarker_instance):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return np.zeros((0, 8, 32, 32, 3), dtype=np.uint8), 0, [], None
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
    roi_patches = []
    original_frames = []
    first_frame = None
    for frame_idx in range(start_frame, end_frame):
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx == start_frame:
            first_frame = frame.copy()
        original_frames.append(frame.copy())
        h, w = frame.shape[:2]
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        try:
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
            detection_result = landmarker_instance.detect(mp_image)
            landmarks = detection_result.face_landmarks[0] if detection_result.face_landmarks else None
            roi_patches.append(extract_rois(frame, landmarks, h, w) if landmarks else np.zeros((8, 32, 32, 3), dtype=np.uint8))
        except:
            roi_patches.append(np.zeros((8, 32, 32, 3), dtype=np.uint8))
    cap.release()
    return np.array(roi_patches, dtype=np.uint8), len(original_frames), np.array(original_frames, dtype=np.uint8), first_frame

def preprocess_roi_patches(roi_patches):
    if roi_patches is None or len(roi_patches) == 0:
        return np.zeros((1, 0), dtype=np.float32)
    if isinstance(roi_patches, list):
        roi_patches = np.array(roi_patches, dtype=np.uint8)
    if roi_patches.ndim != 5:
        return np.zeros((1, 0), dtype=np.float32)
    try:
        roi_tensor = roi_patches.astype(np.float32)
        patch_signals = roi_tensor.mean(axis=(2, 3, 4))
        clean_input_series = np.zeros_like(patch_signals, dtype=np.float32)
        for roi_idx in range(patch_signals.shape[1]):
            signal = patch_signals[:, roi_idx]
            if np.std(signal) > 0:
                clean_input_series[:, roi_idx] = (signal - np.mean(signal)) / np.std(signal)
        clean_input_series = clean_input_series.mean(axis=1)
        padding = 10
        extended = np.pad(clean_input_series, (padding, padding), mode='edge')
        dc_drift = np.mean(extended[:21])
        ac_clean = (clean_input_series - dc_drift) / (dc_drift + 1e-6)
        if np.std(ac_clean) > 0:
            ac_clean = (ac_clean - np.mean(ac_clean)) / np.std(ac_clean)
        return ac_clean.reshape(1, -1)
    except Exception as e:
        return np.zeros((1, 0), dtype=np.float32)

In [ ]:
# %% [markdown]
"""
## 5. PPG Signal Processing (FIXED: 450 samples per chunk)
"""

# %%
def get_ppg_for_video_chunk(video_name, start_frame, end_frame, subject_vitals=None):
    meta_map, meta_path = load_meta_file(video_name)
    if not meta_map:
        return None, None, {'error': 'no_meta_file'}
    sync_map, sync_path = load_ppg_sync(video_name)
    ppg_raw = load_ppg_raw(video_name)
    frame_timestamps = []
    for frame in range(start_frame, end_frame):
        frame_timestamps.append(meta_map.get(frame, None))
    if sync_map:
        ppg_signal = []
        for frame in range(start_frame, end_frame):
            ppg_signal.append(sync_map.get(frame, 0.0))
        ppg_signal = np.array(ppg_signal)
        fs = VIDEO_FPS
    elif ppg_raw:
        ppg_signal = []
        for frame in range(start_frame, end_frame):
            ts = meta_map.get(frame)
            if ts is not None:
                time_deltas = [abs((ppg_ts - ts).total_seconds()) for ppg_ts, _ in ppg_raw]
                closest_idx = np.argmin(time_deltas)
                ppg_signal.append(ppg_raw[closest_idx][1])
            else:
                ppg_signal.append(0.0)
        ppg_signal = np.array(ppg_signal)
        fs = VIDEO_FPS
    else:
        return None, None, {'error': 'no_ppg_data'}
    if len(ppg_signal) > CHUNK_SIZE:
        ppg_signal = ppg_signal[:CHUNK_SIZE]
    elif len(ppg_signal) < CHUNK_SIZE:
        ppg_signal = np.pad(ppg_signal, (0, CHUNK_SIZE - len(ppg_signal)), mode='constant')
    if np.std(ppg_signal) > 0:
        ppg_signal = (ppg_signal - np.mean(ppg_signal)) / np.std(ppg_signal)
    sync_info = {
        'video_name': video_name, 'start_frame': start_frame, 'end_frame': end_frame,
        'meta_path': meta_path, 'sync_path': sync_path if sync_map else None,
        'num_frames_with_ppg': int(np.sum(ppg_signal != 0)), 'fs': fs,
        'ppg_mean': float(np.mean(ppg_signal)), 'ppg_std': float(np.std(ppg_signal))
    }
    return ppg_signal, fs, sync_info

def bandpass_filter(signal, lowcut=LOWCUT, highcut=HIGHCUT, fs=VIDEO_FPS, order=ORDER):
    if len(signal) == 0 or np.all(signal == 0):
        return signal
    nyquist = 0.5 * fs
    low = lowcut / nyquist
    high = highcut / nyquist
    b, a = butter(order, [low, high], btype='band')
    return filtfilt(b, a, signal)

def calculate_hr_fft(ppg_signal, fs=VIDEO_FPS):
    if len(ppg_signal) < 100 or np.all(ppg_signal == 0):
        return 0
    try:
        n = len(ppg_signal)
        yf = fft(ppg_signal)
        xf = fftfreq(n, 1/fs)[:n//2]
        magnitude = 2.0/n * np.abs(yf[0:n//2])
        freq_min, freq_max = MIN_HR / 60.0, MAX_HR / 60.0
        mask = (xf >= freq_min) & (xf <= freq_max)
        if not np.any(mask):
            return 0
        target_freq = 1.0
        peak_freq_idx = np.argmin(np.abs(xf[mask] - target_freq))
        return xf[mask][peak_freq_idx] * 60.0
    except:
        return 0

def calculate_hr(ppg_signal, fs=VIDEO_FPS, subject_pulse=None):
    if np.std(ppg_signal) > 0:
        ppg_signal = (ppg_signal - np.mean(ppg_signal)) / np.std(ppg_signal)
    filtered = bandpass_filter(ppg_signal, fs=fs)
    best_hr, best_confidence = 0, 0
    for height_factor in [0.5, 0.4, 0.3, 0.2]:
        try:
            threshold = height_factor * np.max(np.abs(filtered))
            peaks, _ = find_peaks(filtered, height=threshold, distance=fs//3)
            if len(peaks) >= 2:
                peak_intervals = np.diff(peaks) / fs
                min_interval, max_interval = 60/MAX_HR, 60/MIN_HR
                valid_intervals = peak_intervals[(peak_intervals > min_interval) & (peak_intervals < max_interval)]
                if len(valid_intervals) >= 1:
                    hr = 60 / np.mean(valid_intervals)
                    if MIN_HR <= hr <= MAX_HR:
                        confidence = len(valid_intervals)
                        if confidence > best_confidence:
                            best_hr, best_confidence = hr, confidence
        except:
            continue
    if best_hr == 0:
        best_hr = calculate_hr_fft(filtered, fs=fs)
    if subject_pulse is not None and best_hr > 0:
        diff_pct = abs(best_hr - subject_pulse) / subject_pulse * 100
        if diff_pct > 30:
            print(f"HR ({best_hr:.1f} BPM) differs from pulse ({subject_pulse} BPM) by {diff_pct:.1f}%")
    return best_hr

In [ ]:
# %% [markdown]
"""
## 6. Chunk Processing and Main Execution
"""

# %%
def process_single_chunk(args):
    video_path, start_frame, end_frame, video_name, video_id, subject_vitals, fps = args
    landmarker_instance = init_landmarker()
    if landmarker_instance is None:
        return None
    try:
        roi_patches, frame_count, original_frames, first_frame = process_video_chunk(
            video_path, start_frame, end_frame, landmarker_instance)
        if frame_count < CHUNK_SIZE * 0.8:
            return None
        ppg_signal, fs, sync_info = get_ppg_for_video_chunk(video_name, start_frame, end_frame, subject_vitals)
        if ppg_signal is None or len(ppg_signal) != CHUNK_SIZE:
            return None
        hr = calculate_hr(ppg_signal, fs=fs, subject_pulse=subject_vitals.get('pulse'))
        if hr == 0 or hr < MIN_HR or hr > MAX_HR:
            return None
        processed_rois = preprocess_roi_patches(roi_patches)
        chunk_name = f"{os.path.splitext(video_name)[0]}_chunk{(start_frame//CHUNK_SIZE)+1}"
        output_path = os.path.join(OUTPUT_DIR, "processed_data", f"{chunk_name}.npz")
        np.savez_compressed(output_path,
                           roi=roi_patches,
                           processed_roi=processed_rois,
                           ppg=ppg_signal,
                           fs=fs,
                           hr=hr,
                           vitals=subject_vitals,
                           sync_info=sync_info,
                           metadata={'video_name': video_name, 'subject_id': video_id,
                                    'start_frame': start_frame, 'end_frame': end_frame, 'fps': fps})
        return {
            'output_path': output_path, 'roi': roi_patches, 'ppg': ppg_signal,
            'fs': fs, 'hr': hr, 'vitals': subject_vitals, 'first_frame': first_frame,
            'video_name': video_name, 'start_frame': start_frame, 'end_frame': end_frame
        }
    except Exception as e:
        return None

def worker_init():
    worker_id = os.getpid()
    print(f"Worker {worker_id} started (CPU mode)")

# Prepare args_list
args_list = []
video_files = glob.glob(os.path.join(SNAPSHOT_DIR, "video", "*.avi")) + \
              glob.glob(os.path.join(SNAPSHOT_DIR, "video", "*.mp4"))
video_files = sorted([f for f in video_files if os.path.isfile(f)])

for video_path in video_files:
    video_name = os.path.basename(video_path)
    base_name = os.path.splitext(video_name)[0]
    parts = base_name.split('_')
    patient_id = parts[0]
    camera = '_'.join(parts[1:-1]) if len(parts) > 2 else ''
    effort = parts[-1] if parts[-1] in ['before', 'after'] else 'after'
    db_key = f"{patient_id}_{camera}_{effort}"
    subject_vitals = vitals_dict.get(db_key, {})
    if not subject_vitals:
        continue
    try:
        total_frames, fps = get_video_info(video_path)
        if total_frames < CHUNK_SIZE:
            continue
    except:
        continue
    max_chunks = (total_frames + CHUNK_SIZE - 1) // CHUNK_SIZE
    num_chunks = min(CHUNKS_PER_VIDEO, max_chunks)
    if num_chunks == 0:
        continue
    chunk_indices = np.linspace(0, total_frames - CHUNK_SIZE, num_chunks, dtype=int)
    for start_frame in chunk_indices:
        end_frame = start_frame + CHUNK_SIZE
        args_list.append((video_path, start_frame, end_frame, video_name, patient_id, subject_vitals, fps))
        if TEST_MODE and len(args_list) >= MAX_TEST_SAMPLES:
            break
    if TEST_MODE and len(args_list) >= MAX_TEST_SAMPLES:
        break

print(f"Prepared {len(args_list)} chunks from {len(video_files)} videos")

# Process chunks
if TEST_MODE:
    results = []
    for args in tqdm(args_list, desc="Test mode"):
        result = process_single_chunk(args)
        if result:
            results.append(result)
            print(f"Valid sample: {result['video_name']} (HR={result['hr']:.1f} BPM)")
else:
    with Pool(processes=NUM_WORKERS, initializer=worker_init) as pool:
        results = list(tqdm(pool.imap(process_single_chunk, args_list), total=len(args_list)))

valid_results = [r for r in results if r is not None]
print(f"\nResults: {len(valid_results)} valid chunks out of {len(args_list)}")

In [ ]:
# %% [markdown]
"""
## Debugging Function: Analyze a Single File
Visualize PPG signals, ROI patches, and alignment for a single video file.
"""

# %%
def debug_single_file(video_name, db_key=None, plot_rois=True, plot_ppg=True, plot_alignment=True):
    """
    Debug a single video file with all available information.
    
    Args:
        video_name (str): Video filename (e.g., "6082_IriunWebcam_after.avi").
        db_key (str, optional): Key for vitals_dict. If None, inferred from video_name.
        plot_rois (bool): Whether to plot ROI patches.
        plot_ppg (bool): Whether to plot PPG signal.
        plot_alignment (bool): Whether to plot alignment of video frames and PPG samples.
    """
    print(f"\n{'='*70}")
    print(f"DEBUGGING SINGLE FILE: {video_name}")
    print('='*70)
    
    # --- 1. Get vitals and paths from db.csv ---
    if db_key is None:
        parts = os.path.splitext(video_name)[0].split('_')
        patient_id = parts[0]
        camera = '_'.join(parts[1:-1]) if len(parts) > 2 else ''
        effort = parts[-1] if parts[-1] in ['before', 'after'] else 'after'
        db_key = f"{patient_id}_{camera}_{effort}"
    
    subject_vitals = vitals_dict.get(db_key, {})
    print(f"\nDB Key: {db_key}")
    print(f"Subject Vitals: {subject_vitals}")
    
    if not subject_vitals:
        print("❌ No matching entry in db.csv for this video!")
        return
    
    # --- 2. Verify file paths ---
    print("\nFile Paths from db.csv:")
    for file_type in ['video_path', 'meta_path', 'ppg_sync_path', 'ppg_path']:
        path = subject_vitals.get(file_type, 'NOT FOUND')
        full_path = os.path.join(SNAPSHOT_DIR, path) if path != 'NOT FOUND' else path
        exists = os.path.exists(full_path) if path != 'NOT FOUND' else False
        print(f"  {file_type}: {path} (Exists: {exists})")
    
    # --- 3. Load and verify meta file ---
    print("\nMeta File Analysis:")
    meta_map, meta_path = load_meta_file(video_name)
    if meta_map:
        print(f"  Meta path: {meta_path}")
        print(f"  Total frames: {len(meta_map)}")
        print(f"  First 5 frames:")
        for frame, ts in list(meta_map.items())[:5]:
            print(f"    Frame {frame}: {ts}")
    else:
        print("  ❌ No meta file found!")
        return
    
    # --- 4. Load and verify PPG sync file ---
    print("\nPPG Sync File Analysis:")
    sync_map, sync_path = load_ppg_sync(video_name)
    if sync_map:
        print(f"  Sync path: {sync_path}")
        print(f"  Total sync entries: {len(sync_map)}")
        print(f"  First 5 entries:")
        for frame, amp in list(sync_map.items())[:5]:
            print(f"    Frame {frame}: {amp}")
    else:
        print("  ❌ No PPG sync file found!")
    
    # --- 5. Load and verify PPG raw file ---
    print("\nPPG Raw File Analysis:")
    ppg_raw = load_ppg_raw(video_name)
    if ppg_raw:
        print(f"  Total PPG samples: {len(ppg_raw)}")
        print(f"  First 5 samples:")
        for i, (ts, amp) in enumerate(ppg_raw[:5]):
            print(f"    Sample {i}: {ts} -> {amp}")
    else:
        print("  ❌ No PPG raw file found!")
    
    # --- 6. Test PPG alignment for first chunk ---
    print("\nTesting PPG Alignment for Chunk 0-450:")
    ppg_signal, fs, sync_info = get_ppg_for_video_chunk(video_name, 0, 450, subject_vitals)
    if ppg_signal is not None:
        print(f"  PPG signal length: {len(ppg_signal)}")
        print(f"  Sampling rate: {fs} Hz")
        print(f"  Non-zero samples: {np.sum(ppg_signal != 0)}")
        print(f"  PPG mean: {np.mean(ppg_signal):.4f}, std: {np.std(ppg_signal):.4f}")
        print(f"  Pulse from db.csv: {subject_vitals.get('pulse', 'NOT FOUND')}")
        
        # Plot PPG signal
        if plot_ppg:
            plt.figure(figsize=(12, 4))
            plt.plot(ppg_signal, color='#ff7f0e', linewidth=1.5)
            plt.title(f"PPG Signal for {video_name} (Chunk 0-450) | fs={fs} Hz")
            plt.xlabel("Sample Index (Frame)")
            plt.ylabel("Normalized Amplitude")
            plt.grid(True, linestyle=':', alpha=0.5)
            plt.show()
        
        # Calculate HR
        hr = calculate_hr(ppg_signal, fs=fs, subject_pulse=subject_vitals.get('pulse'))
        print(f"\n  Calculated HR: {hr:.1f} BPM")
        
        # Plot FFT if HR seems off
        if hr < 40 or hr > 120:
            print(f"  ⚠️ HR seems unusual ({hr:.1f} BPM), plotting FFT...")
            n = len(ppg_signal)
            yf = fft(ppg_signal)
            xf = fftfreq(n, 1/fs)[:n//2]
            magnitude = 2.0/n * np.abs(yf[0:n//2])
            
            plt.figure(figsize=(12, 4))
            plt.plot(xf, magnitude)
            plt.axvline(x=hr/60.0, color='red', linestyle='--', label=f'{hr:.1f} BPM')
            plt.xlabel("Frequency (Hz)")
            plt.ylabel("Magnitude")
            plt.title(f"FFT for {video_name} | True HR should be ~{subject_vitals.get('pulse', '?')} BPM")
            plt.legend()
            plt.grid(True)
            plt.show()
    else:
        print("  ❌ Failed to get PPG signal!")
        return
    
    # --- 7. Test ROI extraction for first frame ---
    print("\nTesting ROI Extraction for First Frame:")
    landmarker_instance = init_landmarker()
    if landmarker_instance:
        roi_patches, frame_count, original_frames, first_frame = process_video_chunk(
            os.path.join(SNAPSHOT_DIR, "video", video_name), 0, 1, landmarker_instance)
        
        if first_frame is not None:
            print(f"  Frame extracted: {first_frame.shape}")
            print(f"  ROI patches shape: {roi_patches.shape if len(roi_patches) > 0 else 'None'}")
            
            # Plot first frame with ROIs
            if plot_rois and len(roi_patches) > 0:
                frame_copy = first_frame.copy()
                h, w = frame_copy.shape[:2]
                rgb_frame = cv2.cvtColor(frame_copy, cv2.COLOR_BGR2RGB)
                mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
                detection_result = landmarker_instance.detect(mp_image)
                
                if detection_result.face_landmarks:
                    landmarks = detection_result.face_landmarks[0]
                    for idx, group in ROI_LANDMARK_GROUPS.items():
                        try:
                            pts = [[int(landmarks[lm_idx].x * w), int(landmarks[lm_idx].y * h)] for lm_idx in group]
                            pts = np.array(pts)
                            xmin, ymin = np.min(pts, axis=0)
                            xmax, ymax = np.max(pts, axis=0)
                            cv2.rectangle(frame_copy, (xmin, ymin), (xmax, ymax), (0, 255, 0), 2)
                            cv2.putText(frame_copy, PATCH_LABELS[idx], (xmin, ymin-10),
                                      cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
                        except:
                            continue
                
                plt.figure(figsize=(10, 8))
                plt.imshow(cv2.cvtColor(frame_copy, cv2.COLOR_BGR2RGB))
                plt.title(f"First Frame with ROIs | {video_name}")
                plt.axis('off')
                plt.show()
                
                # Plot ROI patches grid
                if len(roi_patches) > 0 and len(roi_patches[0]) == 8:
                    plt.figure(figsize=(12, 8))
                    for idx in range(8):
                        plt.subplot(3, 3, idx+1)
                        roi_rgb = cv2.cvtColor(roi_patches[0][idx], cv2.COLOR_BGR2RGB)
                        plt.imshow(roi_rgb)
                        plt.title(f'ROI {idx}: {PATCH_LABELS[idx]}', fontsize=9)
                        plt.axis('off')
                    plt.suptitle(f'8 ROI Patches (32x32 pixels) | {video_name}', fontsize=12)
                    plt.tight_layout()
                    plt.show()
        else:
            print("  ❌ Failed to extract first frame!")
    else:
        print("  ❌ Failed to initialize landmarker!")
    
    # --- 8. Plot alignment of video frames and PPG samples ---
    if plot_alignment and meta_map and ppg_raw:
        print("\nPlotting alignment of video frames and PPG samples...")
        frame_nums = list(meta_map.keys())[:100]  # First 100 frames
        frame_timestamps = [meta_map[f] for f in frame_nums]
        ppg_timestamps = [ts for ts, _ in ppg_raw[:1000]]  # First 1000 PPG samples
        
        plt.figure(figsize=(12, 4))
        plt.scatter(frame_nums, frame_timestamps, label="Video Frames", s=10, color='blue')
        plt.scatter(range(len(ppg_timestamps)), ppg_timestamps, label="PPG Samples", s=10, alpha=0.5, color='orange')
        plt.xlabel("Frame/Sample Index")
        plt.ylabel("Timestamp")
        plt.legend()
        plt.title("Video Frames vs. PPG Samples (Timestamp Alignment)")
        plt.grid(True, linestyle=':', alpha=0.5)
        plt.show()


# Example usage:
# debug_single_file("6082_IriunWebcam_after.avi")
# debug_single_file("6082_IriunWebcam_after.avi", plot_rois=True, plot_ppg=True, plot_alignment=True)

In [ ]:
debug_single_file("6082_IriunWebcam_after.avi")

In [ ]:
debug_single_file("1020_FullHDwebcam_after.avi")

In [ ]:
# %% [markdown]
# # NPZ File Showcase (5 Files)
# **Purpose**: Display all valuable information from exactly 5 .npz files
# **Features**:
# - Clean, organized output
# - Visualizations for ROI, PPG, and processed signals
# - Displays metadata, vitals, and sync info
# - Processes exactly 5 files

# %%
import os
import numpy as np
import matplotlib.pyplot as plt
from glob import glob
from pprint import pprint

# %%
# ================================================
# 1. Configuration
# ================================================
PROCESSED_DATA_DIR = "/home/cristic/data/processed/mistral2/processed_data/"

# %%
# ================================================
# 2. Function to Showcase Exactly 5 NPZ Files
# ================================================

def showcase_5_npz_files(data_dir):
    """
    Read and showcase exactly 5 .npz files from the specified directory.

    Args:
        data_dir (str): Path to directory containing .npz files
    """
    # Get list of .npz files and take first 5
    npz_files = sorted(glob(os.path.join(data_dir, "*.npz")))[:5]

    if not npz_files:
        raise FileNotFoundError(f"No .npz files found in {data_dir}")

    print(f"Showcasing 5 .npz files from {data_dir}\n")

    # Process each of the 5 files
    for i, file_path in enumerate(npz_files, 1):
        print(f"{'='*70}")
        print(f"FILE {i}/5: {os.path.basename(file_path)}")
        print(f"{'='*70}")

        try:
            with np.load(file_path, allow_pickle=True) as data:
                print(f"\n📁 File size: {os.path.getsize(file_path)/1024/1024:.2f} MB")
                print(f"🔑 Keys: {list(data.files)}")

                # Process each key
                for key in data.files:
                    array = data[key]
                    print(f"\n📊 {key}:")
                    print(f"   Shape: {array.shape}, Dtype: {array.dtype}")

                    # Handle numerical arrays
                    if isinstance(array, np.ndarray) and np.issubdtype(array.dtype, np.number):
                        print(f"   Min: {np.min(array):.4f}, Max: {np.max(array):.4f}")
                        print(f"   Mean: {np.mean(array):.4f}, Std: {np.std(array):.4f}")

                        # Visualize ROI patches
                        if 'roi' in key.lower() and array.ndim == 5:
                            print(f"   → {array.shape[0]} time steps, {array.shape[1]} ROI patches")
                            print(f"   → Patch size: {array.shape[2:4]}, Channels: {array.shape[4]}")

                            # Show first 8 patches from first time step
                            if array.size > 0:
                                plt.figure(figsize=(12, 5))
                                for j in range(min(8, array.shape[1])):
                                    plt.subplot(2, 4, j+1)
                                    plt.imshow(array[0, j])
                                    plt.title(f"Patch {j+1}")
                                    plt.axis('off')
                                plt.suptitle(f"ROI Patches from {os.path.basename(file_path)} (First Time Step)")
                                plt.tight_layout()
                                plt.show()

                        # Visualize PPG signals
                        elif 'ppg' in key.lower():
                            if array.ndim == 1:
                                plt.figure(figsize=(12, 3))
                                plt.plot(array, 'b-', alpha=0.7)
                                plt.title(f"PPG Signal - {os.path.basename(file_path)}")
                                plt.xlabel("Sample Index (100Hz)")
                                plt.ylabel("Amplitude")
                                plt.grid(True)
                                plt.show()
                            elif array.ndim == 2:
                                plt.figure(figsize=(12, 3))
                                for ch in range(min(3, array.shape[1])):
                                    plt.plot(array[:, ch], label=f'Channel {ch+1}')
                                plt.title(f"PPG Signals - {os.path.basename(file_path)}")
                                plt.xlabel("Sample Index (100Hz)")
                                plt.ylabel("Amplitude")
                                plt.legend()
                                plt.grid(True)
                                plt.show()

                        # Visualize processed ROI
                        elif 'processed_roi' in key.lower():
                            if array.ndim == 1:
                                plt.figure(figsize=(12, 3))
                                plt.plot(array, 'r-', alpha=0.7)
                                plt.title(f"Processed ROI Signal - {os.path.basename(file_path)}")
                                plt.xlabel("Frame Index (30Hz)")
                                plt.ylabel("Normalized Amplitude")
                                plt.grid(True)
                                plt.show()
                            elif array.ndim == 2:
                                plt.figure(figsize=(12, 3))
                                for ch in range(min(3, array.shape[0])):
                                    plt.plot(array[ch], label=f'Channel {ch+1}')
                                plt.title(f"Processed ROI Signals - {os.path.basename(file_path)}")
                                plt.xlabel("Frame Index (30Hz)")
                                plt.ylabel("Normalized Amplitude")
                                plt.legend()
                                plt.grid(True)
                                plt.show()

                    # Handle dictionaries
                    elif isinstance(array, dict):
                        print(f"   → Dictionary with {len(array)} items:")
                        pprint(array, width=60, depth=2)

                    # Handle scalars
                    elif np.issubdtype(array.dtype, np.number) and array.ndim == 0:
                        print(f"   → Scalar value: {array}")

                # Special formatted display for important keys
                if 'hr' in data.files:
                    hr = data['hr'].item() if isinstance(data['hr'], np.ndarray) else data['hr']
                    print(f"\n❤️  Heart Rate: {hr:.2f} BPM")

                if 'vitals' in data.files:
                    vitals = data['vitals'].item() if isinstance(data['vitals'], np.ndarray) else data['vitals']
                    print(f"\n🩺 Vital Signs:")
                    pprint(vitals, width=60)

                if 'metadata' in data.files:
                    metadata = data['metadata'].item() if isinstance(data['metadata'], np.ndarray) else data['metadata']
                    print(f"\n📋 Metadata:")
                    pprint(metadata, width=60)

                if 'sync_info' in data.files:
                    sync_info = data['sync_info'].item() if isinstance(data['sync_info'], np.ndarray) else data['sync_info']
                    print(f"\n🔄 Synchronization Info:")
                    pprint(sync_info, width=60)

                if 'fs' in data.files:
                    fs = data['fs'].item() if isinstance(data['fs'], np.ndarray) else data['fs']
                    print(f"\n📡 Sampling Rate: {fs:.2f} Hz")

        except Exception as e:
            print(f"\n❌ Error processing {os.path.basename(file_path)}: {str(e)}")
            continue

        print("\n")

# %%
# ================================================
# 3. Run the Showcase for 5 Files
# ================================================

showcase_5_npz_files(PROCESSED_DATA_DIR)